In [ ]:
import pandas as pd
import numpy as np
import random
import os

random.seed(42)
np.random.seed(42)

print("Libraries loaded successfully")

In [ ]:
DATA_PATH = "../data/raw/Task_Categories.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

df.head(10)

In [ ]:
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nUnique tasks:")
print(df["Task Description"].nunique())

print("\nUnique categories:")
print(df["Category"].nunique())

print("\nUnique skills:")
print(df["Skill"].nunique())

In [ ]:
print("Categories:")
print(df["Category"].value_counts())

print("\nMost common skills:")
print(df["Skill"].value_counts().head(20))

In [ ]:
df = df.dropna().copy()

df["Task Description"] = (
    df["Task Description"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df["Category"] = (
    df["Category"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df["Skill"] = (
    df["Skill"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df = df.drop_duplicates()

print("Rows after cleaning:", len(df))

df.head()

In [ ]:
tasks = (
    df.groupby(
        ["Task Description", "Category"]
    )["Skill"]
    .apply(lambda x: sorted(set(x)))
    .reset_index()
)

tasks.columns = [
    "task_description",
    "category",
    "required_skills"
]

tasks["skill_count"] = (
    tasks["required_skills"].apply(len)
)

print("Unique tasks:", len(tasks))

tasks.head(10)

In [ ]:
DEPENDENCY_RULES = {
    "frontend": ["ui/ux design", "backend"],
    "backend": ["database", "cloud"],
    "testing": ["frontend", "backend", "ai/ml"],
    "devops": ["backend", "frontend", "testing"],
    "cloud": ["backend", "devops"],
    "ai/ml": ["data science"],
    "data science": ["database"]
}

In [ ]:
PROJECT_CONFIG = {
    "E-Commerce": [
        "frontend", "backend", "database",
        "ui/ux design", "testing", "devops", "cloud"
    ],

    "Social Media": [
        "frontend", "backend", "database",
        "ui/ux design", "ai/ml", "testing", "cloud"
    ],

    "Healthcare": [
        "frontend", "backend", "database",
        "data science", "ai/ml", "testing", "cloud"
    ],

    "Education": [
        "frontend", "backend", "database",
        "ui/ux design", "testing"
    ],

    "Banking": [
        "frontend", "backend", "database",
        "testing", "devops", "cloud"
    ],

    "Food Delivery": [
        "frontend", "backend", "database",
        "ui/ux design", "testing", "cloud"
    ],

    "Project Management": [
        "frontend", "backend", "database",
        "ui/ux design", "testing", "devops"
    ],

    "Booking System": [
        "frontend", "backend", "database",
        "ui/ux design", "testing", "cloud"
    ],

    "Inventory Management": [
        "frontend", "backend", "database",
        "data science", "testing"
    ],

    "Portfolio Website": [
        "frontend", "ui/ux design",
        "testing", "cloud"
    ]
}

In [ ]:
generated_rows = []

NUMBER_OF_PROJECTS = 500

for project_id in range(1, NUMBER_OF_PROJECTS + 1):

    # -------------------------
    # PROJECT TYPE
    # -------------------------

    project_type = random.choice(
        list(PROJECT_CONFIG.keys())
    )

    project_categories = PROJECT_CONFIG[project_type]

    team_size = random.randint(2, 8)

    number_of_tasks = random.randint(8, 20)

    # Only choose tasks suitable for this project type
    project_tasks = tasks[
        tasks["category"].isin(project_categories)
    ]

    selected_tasks = project_tasks.sample(
        n=min(number_of_tasks, len(project_tasks)),
        replace=False
    )

    # Tasks already added to this project
    completed_tasks = []

    for _, task in selected_tasks.iterrows():

        # -------------------------
        # REQUIRED SKILLS
        # -------------------------

        available_skills = task["required_skills"]

        number_of_skills = random.randint(
            1,
            min(4, len(available_skills))
        )

        selected_skills = random.sample(
            available_skills,
            number_of_skills
        )

        # -------------------------
        # DEPENDENCY
        # -------------------------

        possible_dependencies = []

        dependency_categories = DEPENDENCY_RULES.get(
            task["category"],
            []
        )

        for old_task in completed_tasks:

            if old_task["category"] in dependency_categories:

                possible_dependencies.append(
                    old_task["task_description"]
                )

        if possible_dependencies and random.random() < 0.60:

            dependency = random.choice(
                possible_dependencies
            )

        else:

            dependency = "none"

        # -------------------------
        # COMPLEXITY
        # -------------------------

        complexity = random.randint(1, 5)

        # -------------------------
        # ESTIMATED HOURS
        # -------------------------

        estimated_hours = random.randint(
            complexity * 2,
            complexity * 5
        )

        # -------------------------
        # ACTUAL HOURS
        # -------------------------

        variation = random.randint(
            -2,
            complexity * 2
        )

        actual_hours = max(
            1,
            estimated_hours + variation
        )

        # -------------------------
        # PRIORITY
        # -------------------------

        if complexity >= 4:

            priority = random.choice(
                ["high", "high", "medium"]
            )

        elif complexity == 3:

            priority = random.choice(
                ["medium", "high"]
            )

        else:

            priority = random.choice(
                ["low", "medium"]
            )

        # -------------------------
        # DEADLINE
        # -------------------------

        deadline_days = random.randint(
            max(1, complexity),
            complexity * 3
        )

        # -------------------------
        # DELAY
        # -------------------------

        delay_hours = max(
            0,
            actual_hours - estimated_hours
        )

        # -------------------------
        # SAVE TASK
        # -------------------------

        generated_rows.append({

            "project_id": project_id,

            "project_type": project_type,

            "task_description":
                task["task_description"],

            "category":
                task["category"],

            "required_skills":
                ",".join(selected_skills),

            "skill_count":
                len(selected_skills),

            "complexity":
                complexity,

            "priority":
                priority,

            "estimated_hours":
                estimated_hours,

            "actual_hours":
                actual_hours,

            "deadline_days":
                deadline_days,

            "team_size":
                team_size,

            "dependency":
                dependency,

            "delay_hours":
                delay_hours
        })

        # Current task becomes available as
        # a dependency for later tasks
        completed_tasks.append({

            "task_description":
                task["task_description"],

            "category":
                task["category"]
        })


# -------------------------
# CREATE DATAFRAME
# -------------------------

planify_df = pd.DataFrame(generated_rows)

print("Dataset generated successfully")
print("Shape:", planify_df.shape)

planify_df.sample(2)

In [ ]:
planify_df = pd.DataFrame(generated_rows)

print("Final dataset shape:", planify_df.shape)

planify_df.head(2)

In [ ]:
print("Rows:", len(planify_df))

print("\nMissing values:")
print(planify_df.isnull().sum())

print("\nCategories:")
print(planify_df["category"].value_counts())

print("\nPriorities:")
print(planify_df["priority"].value_counts())

print("\nComplexity:")
print(planify_df["complexity"].value_counts())

print("\nEstimated hours:")
print(planify_df["estimated_hours"].describe())

print("\nActual hours:")
print(planify_df["actual_hours"].describe())

In [ ]:
planify_df.sample(2)

In [ ]:
print("========== FINAL DATASET ==========")

print("Shape:", planify_df.shape)

print("\nColumns:")
print(planify_df.columns.tolist())

print("\nMissing values:")
print(planify_df.isnull().sum())

print("\nDuplicate rows:")
print(planify_df.duplicated().sum())

print("\nNumber of projects:")
print(planify_df["project_id"].nunique())

print("\nProject types:")
print(planify_df["project_type"].value_counts())

print("\nCategories:")
print(planify_df["category"].value_counts())

print("\nPriorities:")
print(planify_df["priority"].value_counts())

print("\nComplexity:")
print(planify_df["complexity"].value_counts().sort_index())

In [ ]:
planify_df[
    [
        "skill_count",
        "complexity",
        "estimated_hours",
        "actual_hours",
        "deadline_days",
        "team_size",
        "delay_hours"
    ]
].describe()

In [ ]:
print(
    "Invalid skill counts:",
    len(planify_df[
        (planify_df["skill_count"] < 1) |
        (planify_df["skill_count"] > 4)
    ])
)

print(
    "Invalid complexity:",
    len(planify_df[
        (planify_df["complexity"] < 1) |
        (planify_df["complexity"] > 5)
    ])
)

print(
    "Invalid estimated hours:",
    len(planify_df[
        planify_df["estimated_hours"] <= 0
    ])
)

print(
    "Invalid actual hours:",
    len(planify_df[
        planify_df["actual_hours"] <= 0
    ])
)

print(
    "Invalid deadlines:",
    len(planify_df[
        planify_df["deadline_days"] <= 0
    ])
)

In [ ]:
import matplotlib.pyplot as plt

planify_df["category"].value_counts().plot(
    kind="bar",
    figsize=(10, 5)
)

plt.title("Task Category Distribution")
plt.xlabel("Category")
plt.ylabel("Number of Tasks")
plt.xticks(rotation=45)
plt.show()

planify_df["priority"].value_counts().plot(
    kind="bar",
    figsize=(7, 4)
)

plt.title("Task Priority Distribution")
plt.xlabel("Priority")
plt.ylabel("Number of Tasks")
plt.show()

planify_df["complexity"].value_counts().sort_index().plot(
    kind="bar",
    figsize=(7, 4)
)

plt.title("Task Complexity Distribution")
plt.xlabel("Complexity")
plt.ylabel("Number of Tasks")
plt.show()


planify_df.boxplot(
    column="estimated_hours",
    by="complexity",
    figsize=(8, 5)
)

plt.title("Complexity vs Estimated Hours")
plt.suptitle("")
plt.xlabel("Complexity")
plt.ylabel("Estimated Hours")
plt.show()

In [ ]:
OUTPUT_PATH = "../data/processed/planify_dataset.csv"

planify_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Dataset saved successfully")
print(OUTPUT_PATH)

In [ ]:
from sklearn.model_selection import train_test_split

project_ids = planify_df["project_id"].unique()

train_projects, temp_projects = train_test_split(
    project_ids,
    test_size=0.30,
    random_state=42
)

validation_projects, test_projects = train_test_split(
    temp_projects,
    test_size=0.50,
    random_state=42
)

train_df = planify_df[
    planify_df["project_id"].isin(train_projects)
].copy()

validation_df = planify_df[
    planify_df["project_id"].isin(validation_projects)
].copy()

test_df = planify_df[
    planify_df["project_id"].isin(test_projects)
].copy()




In [ ]:
print("TRAIN")
print("Projects:", train_df["project_id"].nunique())
print("Rows:", len(train_df))

print("\nVALIDATION")
print("Projects:", validation_df["project_id"].nunique())
print("Rows:", len(validation_df))

print("\nTEST")
print("Projects:", test_df["project_id"].nunique())
print("Rows:", len(test_df))

In [ ]:
train_ids = set(train_df["project_id"])
validation_ids = set(validation_df["project_id"])
test_ids = set(test_df["project_id"])

print(
    "Train / Validation overlap:",
    len(train_ids & validation_ids)
)

print(
    "Train / Test overlap:",
    len(train_ids & test_ids)
)

print(
    "Validation / Test overlap:",
    len(validation_ids & test_ids)
)

In [ ]:
train_df.to_csv(
    "../data/processed/train.csv",
    index=False
)

validation_df.to_csv(
    "../data/processed/validation.csv",
    index=False
)

test_df.to_csv(
    "../data/processed/test.csv",
    index=False
)

print("Train, validation and test datasets saved.")

In [ ]:
print("=" * 50)
print("PLANIFY PHASE 2 COMPLETE")
print("=" * 50)

print("Total projects:", planify_df["project_id"].nunique())
print("Total task records:", len(planify_df))
print("Task categories:", planify_df["category"].nunique())

print("\nTraining records:", len(train_df))
print("Validation records:", len(validation_df))
print("Testing records:", len(test_df))

print("\nReady for Phase 3: ML Model Training")